Group name: Stock Success
Created by Lexinejazly Asuncion (017077242), Pranavi Immanni (017207554), Anika Manjesh (017808479)

In [1]:
!pip install pandas seaborn matplotlib


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## **Import Data and Preprocessing**

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Tech
apple_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/apple.csv'
nvidia_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/nvidia.csv'
microsoft_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/microsoft.csv'

# Defense
lockheed_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/lockheedmartin.csv'
northrop_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/northropgrumman.csv'
boeing_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/boeing.csv.zip'

# Waste Management
waste_mgt_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/wastemanagementinc.csv'
republic_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/republicservices.csv'
waste_conn_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/wasteconnections.csv'

In [3]:
url_list = {
    'AAPL': apple_url,
    'NVDA': nvidia_url,
    'MSFT': microsoft_url,
    'LMT': lockheed_url,
    'NOC': northrop_url,
    'BA': boeing_url,
    'WM': waste_mgt_url,
    'RSG': republic_url,
    'WCN': waste_conn_url
}

all_frames = []

rename_map = {
    'date': 'Date',
    'price': 'Close',
    'Price': 'Close',
    'Close/Last': 'Close',
    'close': 'Close',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'volume': 'Volume',
    'Vol.': 'Volume',
    'adj_close': 'Adj Close'
}

tech_tickers = ['AAPL', 'NVDA', 'MSFT']
defense_tickers = ['LMT', 'NOC', 'BA']
waste_tickers = ['WM', 'RSG', 'WCN']

#read csv files and store data into one dataframe
for name, url in url_list.items():
    df = pd.read_csv(url,
                     na_values=["-", ""])

    df.columns = [col.strip() for col in df.columns]

    df = df.rename(columns=rename_map)
    
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.date

    df['Ticker'] = name

    #create column 'Industry' that specifies the industry
    if name in tech_tickers:
        df['Industry'] = 'Tech'
    elif name in defense_tickers:
        df['Industry'] = 'Defense'
    elif name in waste_tickers:
        df['Industry'] = 'Waste Management'

    #keep only the columns that exist in almost all sets
    standard_cols = ['Industry', 'Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
    df = df[standard_cols]

    all_frames.append(df)

#merge the dataframes into one
stocks = pd.concat(all_frames, ignore_index=True)

In [4]:
#clean 'Volume', 'Open', 'High', 'Low', and 'Close' columns
stocks['Volume'] = pd.to_numeric(stocks['Volume'].astype(str).str.replace('M', 'e6').str.replace('K', 'e3').str.replace('B', 'e9'), errors='coerce')

numeric_cols = ['Open', 'High', 'Low', 'Close']

for col in numeric_cols:
    stocks[col] = pd.to_numeric(
        stocks[col].astype(str).str.replace('$', '').str.replace(',', ''), errors='coerce')
     

In [5]:
stocks.head()

,Industry,Date,Ticker,Open,High,Low,Close,Volume
0,Tech,2025-09-26,AAPL,254.10,257.60,253.78,255.46,46080000.0
1,Tech,2025-09-25,AAPL,253.21,257.17,251.71,256.87,55200000.0
2,Tech,2025-09-24,AAPL,255.22,255.74,251.04,252.31,42300000.0
3,Tech,2025-09-23,AAPL,255.88,257.34,253.58,254.43,60280000.0
4,Tech,2025-09-22,AAPL,248.30,256.64,248.12,256.08,105520000.0


In [6]:
stocks.describe()

,Open,High,Low,Close,Volume
count,53011.000000,53011.000000,53011.000000,53011.000000,5.301000e+04
mean,91.936971,92.837901,91.026213,91.950703,1.187463e+08
std,118.605668,119.735844,117.456046,118.614944,2.814343e+08
min,0.060833,0.065667,0.060000,0.061417,7.290000e+04
25%,5.664032,5.728395,5.574074,5.656250,1.366203e+06
50%,43.920000,44.437500,43.312500,43.937500,3.732000e+06
75%,131.250000,132.430000,129.762500,131.135002,6.825630e+07
max,768.850000,774.000000,751.870000,768.020000,9.230856e+09


In [7]:
stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 53011 entries, 0 to 53010
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Industry  53011 non-null  str    
 1   Date      53011 non-null  object 
 2   Ticker    53011 non-null  str    
 3   Open      53011 non-null  float64
 4   High      53011 non-null  float64
 5   Low       53011 non-null  float64
 6   Close     53011 non-null  float64
 7   Volume    53010 non-null  float64
dtypes: float64(5), object(1), str(2)
memory usage: 3.2+ MB


There is missing data for one data entry in the stocks dataframe that corresponds to the Volume column. Since there is only one missing data point, we can remove it.

In [8]:
stocks[stocks['Volume'].isna()]

,Industry,Date,Ticker,Open,High,Low,Close,Volume
52964,Waste Management,2016-06-01,WCN,43.766,43.766,43.766,43.766,NaN


In [9]:
stocks.drop(index=52964, inplace=True)

In [10]:
stocks.info()

<class 'pandas.DataFrame'>
Index: 53010 entries, 0 to 53010
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Industry  53010 non-null  str    
 1   Date      53010 non-null  object 
 2   Ticker    53010 non-null  str    
 3   Open      53010 non-null  float64
 4   High      53010 non-null  float64
 5   Low       53010 non-null  float64
 6   Close     53010 non-null  float64
 7   Volume    53010 non-null  float64
dtypes: float64(5), object(1), str(2)
memory usage: 3.6+ MB


## **Exploratory Data Analysis**

## **Interactive Plot**

# **Machine Learning**

## **Evaluation**